# Tier 1.5b — S8754 stability teachers as corpus rows, with real structures

Prepares and **verifies**; does not train unless `TRAIN_NOW = True`.

Needs `s8754_15b_bundle.zip` and the existing `fireprot_15a_bundle.zip` in `MyDrive/MIPO_NDD/`, plus the 1.5a `fireprot_scores.csv`. Pick a GPU runtime and run the cell.

What it does, in order, and what it refuses to do silently:

1. **Dedupe** S8754 against the FireProt benchmark by containment, then by **domain-level homology** (local alignment, span ≥ 50, identity ≥ 0.40, a benchmark test site inside). Containment alone left 61 homologous proteins in the teacher set; the corpus builder now refuses to run without the homology result.
2. Flip the **inverted sign** (S8754 negative = destabilizing), average pH/temperature replicates, WT-verify every row.
3. Fetch **real coordinates** (RCSB; AlphaFold for UniProt-only entries) and **audit** that the sequence-diff positions match PDB author numbering. The audit aborts on any disagreement.
4. Extract ESM features for the new proteins only, regenerate splits, and check **fold identity**, **no teacher leakage** and **no silent no-op**.
5. Run the **structure-feature probe**: does C-alpha contact number add stability signal beyond ESM LLR on the held-out benchmark?

Background: `S8754_PREFLIGHT.md`, `S8754_HEAD_DECISION.md`, `STRUCTURE_PROBE_RESULTS.md`.


In [ ]:
# Tier 1.5b data preparation: S8754 stability teachers as corpus rows, with real structures.
# Paste this whole file into ONE Colab GPU cell. Re-runnable; resumes feature extraction.
#
# PREREQUISITES on Drive (MyDrive/MIPO_NDD/):
#   s8754_15b_bundle.zip           (this bundle)
#   fireprot_15a_bundle.zip        (already there; supplies the benchmark structures + mipo/fireprot.py)
#   fireprot_15a_results/runs_esm_mlp_seed_42_fold_12/fireprot_scores.csv   (from the 1.5a run)
#
# This cell PREPARES and VERIFIES. It does not train unless TRAIN_NOW is True, and it prints
# the exact training command either way. Freeze the decisions in S8754_HEAD_DECISION.md first.
TRAIN_NOW = False
EXCLUDE = "crop"              # containment dedupe level; the homology dedupe below is ALWAYS applied
TIER = "F_STABILITY_FIELD_TEACHER"
MAX_SPREAD = 2.0              # kcal/mol disagreement cap across pH/temperature rows
WITH_STRUCTURES = True        # RCSB coordinates (AlphaFold for UniProt-only entries)
RUN_PROBE = False             # structure-feature probe: does geometry add to ESM? (uses the new LLRs)
# Runs that collapsed at epoch 0 (constant predictions: the graph models do this on some seeds, as in the bafe
# grid) are never relaunched. Their replacement-seed runs go to runs_reseeded/, outside the main grid.
PARALLEL = "auto"                # concurrent training processes: "auto" = one per 2 vCPUs (max one per run), or an integer
MIN_VARIANTS = 8                 # drop teacher proteins with fewer variants than this (0 = keep all); 8 recommended
TEACHER_STAGES = ["generic_and_legacy_auxiliary", "family_teachers"]   # curriculum stages that receive the teacher tier
FOLD_GENES = ["ASPA", "KCNQ2", "PAX6"]   # test genes to train; folds are looked up BY GENE (v1 ran ASPA and KCNQ2 only)
STAGE_EPOCHS = {"generic_and_legacy_auxiliary": 6, "family_teachers": 4, "verified_ndd": 10}              # None = the v1 schedule (5 + 3 + 8 = 16); or {"stage name": epochs, ...}
COLLAPSED = []   # (model, fold, seed)
RESEEDS = []   # bafe protocol: 42 -> 1042; if that collapses too, 2042
BUNDLE_ZIP = "s8754_15b_bundle.zip"
FIREPROT_ZIP = "fireprot_15a_bundle.zip"
FIREPROT_SCORES = "fireprot_15a_results/runs_esm_mlp_seed_42_fold_12/fireprot_scores.csv"
TAG = "v4_15b_s8754_min8s2_e20"  # names the merged corpus and the run folder

from google.colab import drive

drive.mount("/content/drive")
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd

NDD = Path("/content/drive/MyDrive/MIPO_NDD")
ARCHIVE = NDD / "mipo_ndd_v4_colab_upload.zip"
assert ARCHIVE.exists(), f"missing {ARCHIVE}"
PROJECT = Path("/content") / f"mipo_ndd_v4_validation_{hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()[:12]}"
if not (PROJECT / "mipo/model.py").exists():
    PROJECT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        z.extractall(PROJECT)
assert "branch_probes" in (PROJECT / "mipo/model.py").read_text(), "NOT v3 code"
print("PROJECT:", PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[test]", "biopython", "scikit-learn"],
               check=True, cwd=PROJECT)

digest = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]


def unpack(zip_name, folder):
    path = NDD / zip_name
    assert path.exists(), f"upload {zip_name} to {NDD} first"
    root = Path("/content") / f"unpacked_{folder}"
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None, f"{zip_name} is corrupt"
        z.extractall(root)
    return root / folder


B = unpack(BUNDLE_ZIP, "s8754_15b")
for name in ["prepare_s8754_teachers.py", "build_s8754_corpus.py", "dedupe_s8754_homology.py",
             "build_s8754_structures.py", "audit_s8754_mapping.py", "probe_structure.py",
             "extend_splits_for_teachers.py", "S8754.csv",
             "benchmark_corpus_fireprot.csv", "benchmark_sequences.json"]:
    assert (B / name).exists(), f"bundle missing {name}"
    print(f"  {name:34s} {digest(B / name)}")

# The 1.5a bundle supplies the benchmark's structures and the pooled-metric code, so the probe
# scores with exactly the function 1.5a used.
FP = unpack(FIREPROT_ZIP, "fireprot_15a")
assert digest(FP / "mipo_fireprot.py") == "f6f6199eeef4", "mipo/fireprot.py differs from the 1.5a run"
shutil.copy2(FP / "mipo_fireprot.py", PROJECT / "mipo/fireprot.py")
assert digest(FP / "corpus_fireprot.csv") == digest(B / "benchmark_corpus_fireprot.csv"), \
    "benchmark corpus in the two bundles differs"

WORK = PROJECT / "s8754_15b"
WORK.mkdir(exist_ok=True)
EVAL_GENES = PROJECT / "v4/robust/curriculum_evaluation_genes.csv"
BASE_CORPUS = PROJECT / "v4/robust/curriculum.csv.gz"
BASE_RESOURCES = PROJECT / "v4/resources"
METADATA = PROJECT / "v4/robust/assay_metadata.csv"
for path in [EVAL_GENES, BASE_CORPUS, BASE_RESOURCES / "sequences.json", METADATA]:
    assert path.exists(), f"missing {path}"

# The training recipe is taken from the contrast_pilot_v1 runs themselves rather than rebuilt from a
# config file, so 1.5b differs from the 1.5a baseline only in the data.
V1 = NDD / "contrast_pilot_v1/runs"
V1_PROV = json.loads((V1 / "mipo/seed_42/fold_12/provenance.json").read_text())
CONFIG = json.loads(json.dumps(V1_PROV["config"]))
COMPARABLE = True
WHY = []   # reasons a result is not comparable with v1
base_sha = hashlib.sha256(BASE_CORPUS.read_bytes()).hexdigest()
if base_sha != V1_PROV["corpus_sha256"]:
    COMPARABLE = False
    WHY.append("corpus differs from the one v1 trained on")
    print()
    print("!! WARNING: this archive's base corpus is NOT the corpus contrast_pilot_v1 trained on")
    print(f"   archive {base_sha[:12]} vs pilot {V1_PROV['corpus_sha256'][:12]}. The data preparation below is "
          "still valid, but a 1.5b result cannot be compared with v1 without retraining v1 on this corpus.")
# A tier in no curriculum stage trains on NOTHING with no error, so the stage is patched first
# and the corpus builder asserts it.
for stage_name in TEACHER_STAGES:
    stage = next((x for x in CONFIG["stages"] if x["name"] == stage_name), None)
    assert stage is not None, f"v1 config has no stage {stage_name}: {[x[chr(110) + chr(97) + chr(109) + chr(101)] for x in CONFIG[chr(115) + chr(116) + chr(97) + chr(103) + chr(101) + chr(115)]]}"
    if TIER not in stage["tiers"]:
        stage["tiers"] = stage["tiers"] + [TIER]
SCHEDULE_V1 = {x["name"]: x["epochs"] for x in CONFIG["stages"]}
if STAGE_EPOCHS:
    unknown = set(STAGE_EPOCHS) - set(SCHEDULE_V1)
    assert not unknown, f"STAGE_EPOCHS names unknown stages: {unknown}"
    for x in CONFIG["stages"]:
        x["epochs"] = STAGE_EPOCHS.get(x["name"], x["epochs"])
    if {x["name"]: x["epochs"] for x in CONFIG["stages"]} != SCHEDULE_V1:
        COMPARABLE = False
        WHY.append("schedule differs from v1, so compare against the matched control arm (last cell)")
TOTAL_EPOCHS = sum(x["epochs"] for x in CONFIG["stages"])
CONFIG_PATH = WORK / "config_15b.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + "\n")
print()
for x in CONFIG["stages"]:
    print(f"stage {x['name']:30s} tiers {x['tiers']} (ndd_filter {x.get('ndd_filter')})")

BENCH = ["--benchmark-corpus", str(B / "benchmark_corpus_fireprot.csv"),
         "--benchmark-sequences", str(B / "benchmark_sequences.json")]
GUARDS = ["--corpus-sequences", str(BASE_RESOURCES / "sequences.json"),
          "--evaluation-genes", str(EVAL_GENES), "--require-stage-config", str(CONFIG_PATH), "--tier", TIER]


def run(*args, tail=6000):
    """Run a script and SHOW its output: Colab hides whatever a child process writes to the console."""
    proc = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout[-tail:])
    if proc.returncode:
        print(proc.stderr[-4000:])
        raise RuntimeError(f"{Path(str(args[0])).name} failed with exit code {proc.returncode}")
    return proc.stdout




# 1. Containment dedupe against the benchmark, sign flip, condition averaging, WT verification.
run(B / "prepare_s8754_teachers.py", "--s8754", B / "S8754.csv", *BENCH, "--exclude", EXCLUDE,
    "--max-spread", MAX_SPREAD, "--out", WORK / "teachers")
TEACHERS = WORK / f"teachers/s8754_teachers_{EXCLUDE}.csv"
PARSED = WORK / "teachers/s8754_parsed.csv"

# 2. Containment dedupe misses near-identical constructs and homologs (it left 61 homologous
#    proteins in the teacher set), so the domain-level check is mandatory. Pass 1 only names the
#    candidate teachers; pass 2 rebuilds without the flagged ones.
run(B / "build_s8754_corpus.py", "--teachers", TEACHERS, "--parsed", PARSED, *GUARDS,
    "--allow-no-homology-dedupe", "--allow-no-orientation-check", "--out", WORK / "corpus_pass1")
run(B / "dedupe_s8754_homology.py", "--teacher-sequences", WORK / "corpus_pass1/s8754_sequences.json",
    *BENCH, "--out", WORK / "homology")
run(B / "build_s8754_corpus.py", "--teachers", TEACHERS, "--parsed", PARSED, *GUARDS,
    "--homology-drop", WORK / "homology/s8754_homology_drop.json", "--min-assay-variants", MIN_VARIANTS,
    "--orientation-check", BASE_CORPUS, "--out", WORK / "corpus")
new_rows = pd.read_csv(WORK / "corpus/s8754_corpus_rows.csv")
new_seqs = json.loads((WORK / "corpus/s8754_sequences.json").read_text())
print(f"\nteacher set after all dedupe: {len(new_rows)} variants, {len(new_seqs)} proteins")

# 3. Merge into new resources, leaving the originals byte-identical.
RESOURCES = WORK / "resources"
if not (RESOURCES / "sequences.json").exists():
    shutil.copytree(BASE_RESOURCES, RESOURCES, dirs_exist_ok=True)
merged_seqs = json.loads((BASE_RESOURCES / "sequences.json").read_text())
assert not (set(merged_seqs) & set(new_seqs)), "gene id collision with existing corpus"
merged_seqs.update(new_seqs)
(RESOURCES / "sequences.json").write_text(json.dumps(merged_seqs, indent=2) + "\n")

# 4. Real coordinates, then an independent audit of the position mapping against them.
if WITH_STRUCTURES:
    run(B / "build_s8754_structures.py", "--sequences", WORK / "corpus/s8754_sequences.json",
        "--manifest", WORK / "corpus/s8754_structure_manifest.csv", "--out", RESOURCES,
        "--pdb-cache", WORK / "pdb")
    run(B / "audit_s8754_mapping.py", "--parsed", PARSED, "--sequences", WORK / "corpus/s8754_sequences.json",
        "--structures", RESOURCES, "--out", WORK / "audit")   # exits nonzero on any disagreement

base = pd.read_csv(BASE_CORPUS, compression="gzip", low_memory=False)
assert not (set(new_rows.gene) & set(base.gene)), "gene name collision with existing corpus"
CORPUS = WORK / f"curriculum_{TAG}.csv.gz"
merged = pd.concat([base, new_rows], ignore_index=True)
import gzip
# gzip embeds a timestamp and the filename, so a plain to_csv(compression="gzip") gives a NEW sha every
# session. Resume compares that sha, so a Colab recycle would orphan every partial run. Pin both.
with open(CORPUS, "wb") as raw, gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz:
    merged.to_csv(gz, index=False)
print(f"merged corpus: {len(base)} + {len(new_rows)} = {len(merged)} rows | "
      f"{base.gene.nunique()} + {new_rows.gene.nunique()} = {merged.gene.nunique()} genes | sha {digest(CORPUS)}")

# 5. ESM features for the new proteins only; existing cache entries are skipped.
DRIVE_CACHE = NDD / "v4_650M_masked"
CACHE = PROJECT / "v4_650M_masked"
assert (DRIVE_CACHE / "manifest.json").exists(), f"missing {DRIVE_CACHE}"
if not (CACHE / "manifest.json").exists():
    shutil.copytree(DRIVE_CACHE, CACHE, dirs_exist_ok=True)
before = {p.name for p in CACHE.iterdir() if p.is_dir()}
print(f"\nfeature cache: {len(before)} proteins present; extracting the "
      f"{sum(1 for v in new_seqs.values() if v['key'] not in before)} new ones "
      f"({sum(len(v['sequence']) for v in new_seqs.values())} residues)")
run("-m", "mipo", "features", "--corpus", CORPUS, "--resources", RESOURCES, "--cache", CACHE)
added = sorted({p.name for p in CACHE.iterdir() if p.is_dir()} - before)
for key in added:  # persist so a later session does not re-extract
    shutil.copytree(CACHE / key, DRIVE_CACHE / key, dirs_exist_ok=True)
print(f"mirrored {len(added)} new cache entries to Drive")

# 6. Splits, built the way the pilot's were: the base corpus's source split is expanded to 5
#    validation / 3 calibration groups (seed 42), CHECKED against the folds contrast_pilot_v1 actually
#    used, and only then extended with the teacher genes (train role only).
SOURCE_SPLITS = PROJECT / "v4/robust/curriculum_splits.json"
assert SOURCE_SPLITS.exists(), f"missing {SOURCE_SPLITS}"
BASE_EXPANDED = WORK / "splits_base_expanded.json"
run(PROJECT / "scripts/prepare_validation_splits.py", "--corpus", BASE_CORPUS, "--source", SOURCE_SPLITS,
    "--out", BASE_EXPANDED, "--validation-groups", 5, "--calibration-groups", 3, "--seed", 42)
base_split = json.loads(BASE_EXPANDED.read_text())
PILOT_FOLDS = []
for gene in FOLD_GENES:
    hits = [i for i, f in enumerate(base_split["folds"]) if f["test"] == [gene]]
    assert len(hits) == 1, f"test gene {gene} matches {len(hits)} folds"
    PILOT_FOLDS.append(hits[0])
for gene, fold in zip(FOLD_GENES, PILOT_FOLDS):
    mine = base_split["folds"][fold]
    n_val, n_cal = len(mine["validation"]), len(mine["calibration"])
    v1_file = V1 / f"mipo/seed_42/fold_{fold:02d}/provenance.json"
    if v1_file.exists():
        pv = json.loads(v1_file.read_text())["fold"]
        for role in ["test", "validation", "calibration"]:
            assert mine[role] == pv[role], (f"fold {fold} {role} differs from contrast_pilot_v1 "
                                            f"({mine[role]} vs {pv[role]}); results would not be comparable")
        print(f"fold {fold} ({gene}): {n_val} validation + {n_cal} calibration genes match contrast_pilot_v1 exactly")
    else:
        print(f"fold {fold} ({gene}): {n_val} validation + {n_cal} calibration genes; v1 never ran this fold, so use the control arm")
SPLITS = WORK / f"splits_{TAG}.json"
run(B / "extend_splits_for_teachers.py", "--splits", BASE_EXPANDED, "--corpus", CORPUS,
    "--teacher-genes", WORK / "corpus/s8754_sequences.json", "--mipo-src", PROJECT, "--out", SPLITS)
ext = json.loads(SPLITS.read_text())
leak = [g for f in ext["folds"] for role in ["test", "validation", "calibration"] for g in f[role]
        if g.startswith("S8754_")]
assert not leak, f"S8754 genes reached a held-out split: {sorted(set(leak))[:5]}"
print(f"folds: {len(ext['folds'])}; held-out roles identical to the pilot-style base splits; "
      f"{len(new_seqs)} teacher genes added to train only; no S8754 gene in any held-out split")

# 7. The rows must actually be selected by a stage. A silent no-op is the failure mode.
sys.path.insert(0, str(PROJECT))
from mipo.corpus import read_corpus
from mipo.train import select_stage_rows

d = read_corpus(CORPUS)
for st in CONFIG["stages"]:
    rows = select_stage_rows(d, st)
    mine = int((rows.supervision_tier == TIER).sum())
    print(f"  stage {st['name']:32s} rows {len(rows):7d} | S8754 rows {mine}")
    if st["name"] in TEACHER_STAGES:
        assert mine == len(new_rows), f"stage selects {mine} of {len(new_rows)} S8754 rows"

# 7b. Everything below must be byte-identical across Colab sessions: resume compares the sha of the
#     corpus, splits and sequences and the digest of every feature and structure file. Record the
#     fingerprint on the first preparation; later sessions must reproduce it exactly or stop.
full = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
new_keys = sorted(v["key"] for v in new_seqs.values())
struct = lambda k: RESOURCES / "structures" / f"{k}.npz"
FINGERPRINT = {
    "corpus": full(CORPUS), "splits": full(SPLITS), "sequences": full(RESOURCES / "sequences.json"),
    "config": full(CONFIG_PATH),
    "structures": hashlib.sha256("".join(f"{k}:{full(struct(k)) if struct(k).exists() else '-'}"
                                         for k in new_keys).encode()).hexdigest(),
    "features": hashlib.sha256("".join(f"{k}:{full(CACHE / k / 'esm.npy')}:{full(CACHE / k / 'llr.npy')}"
                                       for k in new_keys).encode()).hexdigest()}
(NDD / TAG).mkdir(exist_ok=True)
FP_FILE = NDD / TAG / "prepared_inputs.json"
if FP_FILE.exists():
    recorded = json.loads(FP_FILE.read_text())
    drift = sorted(k for k in FINGERPRINT if recorded.get(k) != FINGERPRINT[k])
    if drift:
        raise RuntimeError(f"Prepared inputs differ from the first preparation in: {drift}. Training or resuming from "
                           f"them would fail resume provenance. Use a new TAG, or delete {FP_FILE} only if nothing "
                           "has trained yet.")
    print("prepared inputs identical to the first preparation (resume-safe)")
else:
    FP_FILE.write_text(json.dumps(FINGERPRINT, indent=2) + chr(10))
    print(f"recorded prepared-input fingerprint -> {FP_FILE}")
print("  corpus sha", FINGERPRINT["corpus"][:12], "| splits", FINGERPRINT["splits"][:12])
PLAN = {"folds": PILOT_FOLDS, "fold_genes": FOLD_GENES, "epochs": TOTAL_EPOCHS, "teacher_stages": TEACHER_STAGES}
(NDD / TAG / "plan.json").write_text(json.dumps(PLAN, indent=2) + chr(10))

# 8. Does geometry add to ESM? Fit on the (deduped) teachers, score the FireProt benchmark.
if RUN_PROBE and WITH_STRUCTURES:
    SCORES = NDD / FIREPROT_SCORES
    assert SCORES.exists(), f"missing {SCORES}; needed for the benchmark LLR and the trained-mu reference"
    run(B / "probe_structure.py", "--teacher-rows", WORK / "corpus/s8754_corpus_rows.csv",
        "--teacher-sequences", WORK / "corpus/s8754_sequences.json", "--teacher-structures", RESOURCES,
        "--benchmark-corpus", FP / "corpus_fireprot.csv",
        "--benchmark-sequences", FP / "resources_fireprot/sequences.json",
        "--benchmark-structures", FP / "resources_fireprot/structures",
        "--teacher-cache", CACHE, "--benchmark-scores", SCORES, "--mipo-src", PROJECT,
        "--bootstrap", 500, "--out", WORK / "probe")
    shutil.copytree(WORK / "probe", NDD / f"{TAG}_probe", dirs_exist_ok=True)

BASE_CMD = [sys.executable, str(PROJECT / "scripts/run_experiments.py"),
            "--corpus", str(CORPUS), "--resources", str(RESOURCES), "--cache", str(CACHE),
            "--splits", str(SPLITS), "--config", str(CONFIG_PATH), "--metadata", str(METADATA),
            "--out", str(NDD / TAG / "runs"), "--seeds", "42"]
JOBS = ([(m, f, 42, "runs") for m in ("mipo", "esm_mlp") for f in PILOT_FOLDS if (m, f, 42) not in COLLAPSED]
        + [(m, f, sd, "runs_reseeded") for m, f, sd in RESEEDS])


def job_cmd(model, fold, seed, sub):
    cmd = list(BASE_CMD)
    cmd[cmd.index("--seeds") + 1] = str(seed)
    cmd[cmd.index("--out") + 1] = str(NDD / TAG / sub)
    return cmd + ["--folds", str(fold), "--ablations", model]


def set_aside_stale(tag):
    """train.py resumes a partial run only on the torch/Python that started it, and cannot read a half-written
    last.pt. Such a run is moved aside (kept, never deleted) and restarts from epoch 0."""
    import platform
    import time as _t
    import torch
    runtime = [str(torch.__version__), platform.python_version()]
    root = NDD / tag
    for last in [q for sub in ("runs", "runs_reseeded") for q in sorted((root / sub).glob("*/seed_*/fold_*/last.pt"))]:
        partial = last.parent
        if (partial / "test_summary.json").exists():
            continue
        try:
            saved = torch.load(last, map_location="cpu", weights_only=True)["provenance"]
            stale = [saved["torch"], saved["python"]] != runtime
            old_torch = saved["torch"]
            why = f"started on torch {old_torch}, runtime is torch {runtime[0]}"
        except Exception as error:  # e.g. half-written when the runtime dropped
            stale, why = True, f"last.pt unreadable ({type(error).__name__})"
        if stale:
            stamp = _t.strftime("%Y%m%d-%H%M%S")
            target = root / "set_aside_partial_runs" / partial.relative_to(root).parent / f"{partial.name}_{stamp}"
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(partial), str(target))
            print(f"set aside {partial.name}: {why}; that run restarts. Kept at {target}")


def launch_lanes(jobs, cmd_fn, log_dir):
    """One bash lane per concurrent slot; a lane runs its jobs one after another, output going to per-run logs."""
    import os
    import shlex
    import time
    cores = os.cpu_count() or 1
    lanes = min(len(jobs), max(1, cores // 2)) if PARALLEL == "auto" else min(len(jobs), int(PARALLEL))
    threads = max(1, cores // lanes)
    env = {**os.environ, "OMP_NUM_THREADS": str(threads), "MKL_NUM_THREADS": str(threads), "PYTHONUNBUFFERED": "1"}
    log_dir.mkdir(parents=True, exist_ok=True)
    for k, group in enumerate([jobs[i::lanes] for i in range(lanes)]):   # mipo and esm_mlp spread across lanes
        steps = []
        for model, fold, seed, sub in group:
            name = f"{model}_fold{fold:02d}" if seed == 42 else f"{model}_seed{seed}_fold{fold:02d}"
            steps.append(shlex.join(cmd_fn(model, fold, seed, sub)) + " >> " + shlex.quote(str(log_dir / f"{name}.log")) + " 2>&1")
        proc = subprocess.Popen(["bash", "-c", " ; ".join(steps)], env=env, start_new_session=True)
        print(f"lane {k}: pid {proc.pid} runs, in order: " + ", ".join(f"{m} fold {f} seed {sd}" for m, f, sd, _ in group))
        time.sleep(30)  # stagger, so the data pipelines do not all start on the CPU together
    return lanes, threads


print("=" * 70)
print("PREPARATION COMPLETE. One background process per run, the way the v1 pilot was launched:")
for model, fold, seed, sub in JOBS:
    print("  " + " ".join(job_cmd(model, fold, seed, sub)))
print()
stage_epochs = [(x["name"], x["epochs"]) for x in CONFIG["stages"]]
print(f"schedule: {stage_epochs} = {TOTAL_EPOCHS} epochs | folds {dict(zip(FOLD_GENES, PILOT_FOLDS))}")
print("recipe: config and splits taken from contrast_pilot_v1; "
      + ("COMPARABLE with v1." if COMPARABLE else "NOT comparable with v1: " + "; ".join(WHY)))
import os
cores = os.cpu_count() or 1
ram_gb = int([l for l in open("/proc/meminfo") if l.startswith("MemTotal")][0].split()[1]) // 1048576
lanes = min(len(JOBS), max(1, cores // 2)) if PARALLEL == "auto" else min(len(JOBS), int(PARALLEL))
print(f"hardware: {cores} vCPUs, {ram_gb} GB RAM -> {lanes} concurrent training process(es), {max(1, cores // lanes)} thread(s) each")
if cores < 4:
    print("!! Under 4 vCPUs: each run needs 1-2 cores (single-threaded data pipeline), so runs go one at a time and the")
    print("   total is about 4x longer. Choose Runtime -> Change runtime type -> High-RAM for more vCPUs, then re-run.")

if TRAIN_NOW:
    running = subprocess.run(["pgrep", "-af", "run_experiments"], capture_output=True, text=True).stdout.strip()
    assert not running, "run_experiments is already running; not launching a second copy: " + running
    set_aside_stale(TAG)
    launch_lanes(JOBS, job_cmd, NDD / TAG / "logs")
    print()
    print("Runs continue in the background after this cell ends, but NOT across a runtime recycle.")
    print("After a recycle, re-run this cell with TRAIN_NOW=True: finished runs are skipped and partial runs resume.")
    print("If a log shows a resume provenance mismatch, the Colab image changed (torch/python); see the v1 resume notebook.")
else:
    print("TRAIN_NOW=False: nothing trained. Freeze the decisions in S8754_HEAD_DECISION.md first, then set "
          "TRAIN_NOW=True and re-run this cell (preparation is cached and resumes quickly).")


In [ ]:
# STATUS: run any time while the 1.5b runs train. Reads Drive only; touches nothing.
import json
from pathlib import Path

NDD = Path("/content/drive/MyDrive/MIPO_NDD")
TAG = "v4_15b_s8754_min8s2_e20"
BASE, LOGS = NDD / TAG, NDD / TAG / "logs"
for log in sorted(LOGS.glob("*.log")) + sorted((BASE / "logs_reseeded").glob("*.log")):
    tail = log.read_text(errors="ignore").replace(chr(13), chr(10)).strip().splitlines()[-1:]
    print(f"{log.name:24s} {(tail[0] if tail else chr(45))[-150:]}")
print()
for d in [q for sub in ("runs", "runs_reseeded") for q in sorted((BASE / sub).glob("*/seed_*/fold_*"))]:
    state = "FINISHED" if (d / "test_summary.json").exists() else ("partial" if (d / "last.pt").exists() else "started")
    h = d / "history.json"
    epochs = len(json.loads(h.read_text())) if h.exists() else 0
    print(f"{state:9s} {str(d.relative_to(BASE)):38s} epochs logged: {epochs}")


In [ ]:
# WAIT: keeps this notebook busy (so Colab does not treat it as idle) and reports every POLL_S seconds
# until all four runs finish. Safe to interrupt any time: the runs are separate background processes.
import json
import subprocess
import time
from pathlib import Path

POLL_S, MAX_HOURS = 300, 8
NDD = Path("/content/drive/MyDrive/MIPO_NDD")
TAG = "v4_15b_s8754_min8s2_e20"
BASE = NDD / TAG
plan = json.loads((BASE / "plan.json").read_text()) if (BASE / "plan.json").exists() else {"folds": [12, 57]}
FOLDS = plan["folds"]
TOTAL = "/" + str(plan["epochs"]) if plan.get("epochs") else ""
COLLAPSED = []   # keep in step with the main cell
RESEEDS = []
EXPECTED = ([f"runs/{m}/seed_42/fold_{f:02d}" for m in ("mipo", "esm_mlp") for f in FOLDS if (m, f, 42) not in COLLAPSED]
            + [f"runs_reseeded/{m}/seed_{sd}/fold_{f:02d}" for m, f, sd in RESEEDS])


def describe(rel):
    d = BASE / rel
    if (d / "test_summary.json").exists():
        return "FINISHED", True
    h = d / "history.json"
    if not h.exists():
        return "no epoch finished yet", False
    hist = json.loads(h.read_text())
    last = hist[-1]
    val = [x["validation_spearman"] for x in hist]
    note = ""
    if len(val) >= 2 and max(abs(v) for v in val) < 0.02:
        note = "  <-- COLLAPSED? validation Spearman is ~0 at every epoch; this run is probably dead"
    return f"epochs done {last['epoch'] + 1}{TOTAL} stage {last['stage']} val {last['validation_spearman']:+.3f}{note}", False


start = time.time()
while True:
    states = {rel: describe(rel) for rel in EXPECTED}
    alive = subprocess.run(["pgrep", "-fc", "run_experiments"], capture_output=True, text=True).stdout.strip()
    print(time.strftime("%H:%M:%S"), f"| run_experiments processes alive: {alive}")
    for rel, (text, _) in states.items():
        print(f"   {rel:40s} {text}")
    if all(done for _, done in states.values()):
        print("ALL EXPECTED RUNS FINISHED. Next: run fireprot_15b_score.ipynb.")
        break
    if alive in ("", "0"):
        logs = sorted((BASE / "logs").glob("*.log")) if (BASE / "logs").exists() else []
        if not logs:
            print("Nothing has been launched yet: no logs exist. In the main cell set TRAIN_NOW = True and run it,")
            print("then start this cell again.")
        else:
            print("NO run_experiments process is alive but runs are unfinished: the runtime was recycled or the jobs crashed.")
            for log in logs:
                tail = log.read_text(errors="ignore").replace(chr(13), chr(10)).strip().splitlines()[-4:]
                print("--- " + log.name)
                print(chr(10).join(line[:200] for line in tail))
            print("If a log ends in a Traceback, send it to me. Otherwise re-run the main cell with TRAIN_NOW = True:")
            print("finished runs are skipped and partial runs resume.")
        break
    if time.time() - start > MAX_HOURS * 3600:
        print(f"Stopped waiting after {MAX_HOURS} h; the runs are still going. Re-run this cell to keep watching.")
        break
    time.sleep(POLL_S)


In [ ]:
# CONTROL ARM: the same schedule, folds, seeds and config as the treatment, but WITHOUT the S8754 teachers.
# It is the only baseline that shares the treatment's schedule and fold set, so T1 and T3 need it whenever the
# main cell says "NOT comparable with v1". Run it in the SAME session as the main cell: it reuses that cell's variables.
RUN_CONTROL = False
START_WHILE_TREATMENT_RUNS = False   # True = share the CPUs with a running treatment (both slow down)

import json
import subprocess

assert "launch_lanes" in globals(), "Run the main cell first (TRAIN_NOW = False is enough), in this same session."
CTRL_TAG = TAG + "_control"
ctrl_config = json.loads(CONFIG_PATH.read_text())
for x in ctrl_config["stages"]:
    x["tiers"] = [t for t in x["tiers"] if t != TIER]
CTRL_CONFIG = WORK / "config_control.json"
CTRL_CONFIG.write_text(json.dumps(ctrl_config, indent=2) + chr(10))
CTRL_BASE = [sys.executable, str(PROJECT / "scripts/run_experiments.py"),
             "--corpus", str(BASE_CORPUS), "--resources", str(BASE_RESOURCES), "--cache", str(CACHE),
             "--splits", str(BASE_EXPANDED), "--config", str(CTRL_CONFIG), "--metadata", str(METADATA),
             "--out", str(NDD / CTRL_TAG / "runs"), "--seeds", "42"]
CTRL_JOBS = [(m, f, 42, "runs") for m in ("mipo", "esm_mlp") for f in PILOT_FOLDS]


def ctrl_cmd(model, fold, seed, sub):
    return list(CTRL_BASE) + ["--folds", str(fold), "--ablations", model]


(NDD / CTRL_TAG).mkdir(exist_ok=True)
(NDD / CTRL_TAG / "plan.json").write_text(json.dumps({**PLAN, "teacher_stages": []}, indent=2) + chr(10))
print(f"control arm: {len(CTRL_JOBS)} runs, {TOTAL_EPOCHS} epochs each, folds {dict(zip(FOLD_GENES, PILOT_FOLDS))}, no teachers")
print("  corpus", BASE_CORPUS.name, "| splits", BASE_EXPANDED.name, "| results ->", NDD / CTRL_TAG)
if RUN_CONTROL:
    running = subprocess.run(["pgrep", "-af", "run_experiments"], capture_output=True, text=True).stdout.strip()
    assert not any(CTRL_TAG in ln for ln in running.splitlines()), "the control arm is already running"
    assert START_WHILE_TREATMENT_RUNS or not running, "training is already running; wait for it, or set START_WHILE_TREATMENT_RUNS = True: " + running
    set_aside_stale(CTRL_TAG)
    launch_lanes(CTRL_JOBS, ctrl_cmd, NDD / CTRL_TAG / "logs")
    print("To watch it, set TAG in the monitor cell to " + CTRL_TAG)
else:
    print("RUN_CONTROL = False: nothing launched. Set it to True once the treatment has finished.")
